# 25 Text Classification using fastText

In [1]:
import pandas as pd

df = pd.read_csv("dataset/ecommerce_dataset.csv", names=['category', 'descriptions'], header=None)
print(df.shape)
df.head()

(50425, 2)


,category,descriptions
0,Household,Paper Plane Design Framed Wall Hanging Motivat...
1,Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ..."
2,Household,SAF 'UV Textured Modern Art Print Framed' Pain...
3,Household,"SAF Flower Print Framed Painting (Synthetic, 1..."
4,Household,Incredible Gifts India Wooden Happy Birthday U...


In [2]:
df.category.value_counts()

category
Household                 19313
Books                     11820
Electronics               10621
Clothing & Accessories     8671
Name: count, dtype: int64

In [3]:
df.dropna(inplace=True)
df.shape

(50424, 2)

In [4]:
df.category.replace("Clothing & Accessories", "Clothing_Accessories", inplace=True)
df.category.unique()

/var/folders/26/lymd6t5976g6901lvz279yv80000gn/T/ipykernel_42726/3168945168.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df.category.replace("Clothing & Accessories", "Clothing_Accessories", inplace=True)


array(['Household', 'Books', 'Clothing_Accessories', 'Electronics'],
      dtype=object)

In [6]:
df.category.value_counts()

category
Household               19313
Books                   11820
Electronics             10621
Clothing_Accessories     8670
Name: count, dtype: int64

In [8]:
df["category"] = "__label__" + df["category"].astype(str)
df.head()

,category,descriptions
0,__label__Household,Paper Plane Design Framed Wall Hanging Motivat...
1,__label__Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ..."
2,__label__Household,SAF 'UV Textured Modern Art Print Framed' Pain...
3,__label__Household,"SAF Flower Print Framed Painting (Synthetic, 1..."
4,__label__Household,Incredible Gifts India Wooden Happy Birthday U...


In [21]:
df['category_description'] = df['category'] + " " + df['descriptions']
df.head()

,category,descriptions,category_description
0,__label__Household,Paper Plane Design Framed Wall Hanging Motivat...,__label__Household Paper Plane Design Framed W...
1,__label__Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ...",__label__Household SAF 'Floral' Framed Paintin...
2,__label__Household,SAF 'UV Textured Modern Art Print Framed' Pain...,__label__Household SAF 'UV Textured Modern Art...
3,__label__Household,"SAF Flower Print Framed Painting (Synthetic, 1...",__label__Household SAF Flower Print Framed Pai...
4,__label__Household,Incredible Gifts India Wooden Happy Birthday U...,__label__Household Incredible Gifts India Wood...


In [22]:
df['category_description'][0]

'__label__Household Paper Plane Design Framed Wall Hanging Motivational Office Decor Art Prints (8.7 X 8.7 inch) - Set of 4 Painting made up in synthetic frame with uv textured print which gives multi effects and attracts towards it. This is an special series of paintings which makes your wall very beautiful and gives a royal touch. This painting is ready to hang, you would be proud to possess this unique painting that is a niche apart. We use only the most modern and efficient printing technology on our prints, with only the and inks and precision epson, roland and hp printers. This innovative hd printing technique results in durable and spectacular looking prints of the highest that last a lifetime. We print solely with top-notch 100% inks, to achieve brilliant and true colours. Due to their high level of uv resistance, our prints retain their beautiful colours for many years. Add colour and style to your living space with this digitally printed painting. Some are for pleasure and so

In [23]:
import re

text = df['category_description'][0]

def preprocess(text):
    text = re.sub(r'[^\w\s\']', ' ', text)
    text = re.sub(r' +', " ", text)
    return text.strip().lower()

In [24]:
preprocess(text)

'__label__household paper plane design framed wall hanging motivational office decor art prints 8 7 x 8 7 inch set of 4 painting made up in synthetic frame with uv textured print which gives multi effects and attracts towards it this is an special series of paintings which makes your wall very beautiful and gives a royal touch this painting is ready to hang you would be proud to possess this unique painting that is a niche apart we use only the most modern and efficient printing technology on our prints with only the and inks and precision epson roland and hp printers this innovative hd printing technique results in durable and spectacular looking prints of the highest that last a lifetime we print solely with top notch 100 inks to achieve brilliant and true colours due to their high level of uv resistance our prints retain their beautiful colours for many years add colour and style to your living space with this digitally printed painting some are for pleasure and some for eternal bli

In [25]:
df['category_description'] = df['category_description'].map(preprocess)

In [26]:
df.head()

,category,descriptions,category_description
0,__label__Household,Paper Plane Design Framed Wall Hanging Motivat...,__label__household paper plane design framed w...
1,__label__Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ...",__label__household saf 'floral' framed paintin...
2,__label__Household,SAF 'UV Textured Modern Art Print Framed' Pain...,__label__household saf 'uv textured modern art...
3,__label__Household,"SAF Flower Print Framed Painting (Synthetic, 1...",__label__household saf flower print framed pai...
4,__label__Household,Incredible Gifts India Wooden Happy Birthday U...,__label__household incredible gifts india wood...


In [27]:
df.category_description

0        __label__household paper plane design framed w...
1        __label__household saf 'floral' framed paintin...
2        __label__household saf 'uv textured modern art...
3        __label__household saf flower print framed pai...
4        __label__household incredible gifts india wood...
                               ...                        
50420    __label__electronics strontium microsd class 1...
50421    __label__electronics crossbeats wave waterproo...
50422    __label__electronics karbonn titanium wind w4 ...
50423    __label__electronics samsung guru fm plus sm b...
50424    __label__electronics micromax canvas win w121 ...
Name: category_description, Length: 50424, dtype: object

In [28]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(
    df,
    test_size=0.2
)

In [29]:
train.shape

(40339, 3)

In [31]:
test.shape

(10085, 3)

In [32]:
train.head()

,category,descriptions,category_description
21740,__label__Books,Practice Sets for JEE Advanced 2019 About the ...,__label__books practice sets for jee advanced ...
1343,__label__Household,SinghsVillas Decor 100% Cotton Barbie Bedsheet...,__label__household singhsvillas decor 100 cott...
28060,__label__Books,Art Ocean A5 Drawing Book Pack of 2 A5 Drawing...,__label__books art ocean a5 drawing book pack ...
28014,__label__Books,"Prahaas Drawing Book with Butter Paper, 15 X 1...",__label__books prahaas drawing book with butte...
19119,__label__Household,Techsun Plastic Home Tool Box Set with Removab...,__label__household techsun plastic home tool b...


In [33]:
test.head()

,category,descriptions,category_description
19083,__label__Household,Stanley 1320-Watt 125mm Tile Cutter (Yellow an...,__label__household stanley 1320 watt 125mm til...
3258,__label__Household,AAI Heavy-duty Stainless Steel Connection Pipe...,__label__household aai heavy duty stainless st...
36682,__label__Clothing_Accessories,Ultra Zon Women's Silicone Adhesive Stick Push...,__label__clothing_accessories ultra zon women'...
38713,__label__Clothing_Accessories,CREATURE Matt Finish Club Master Wayfarer Uv P...,__label__clothing_accessories creature matt fi...
3686,__label__Household,ExclusiveLane 'The Earthern Bottle' Tea-Light ...,__label__household exclusivelane 'the earthern...


In [34]:
train.to_csv("dataset/ecommerce.train", columns=["category_description"], index=False, header = False)
test.to_csv("dataset/ecommerce.test", columns=["category_description"], index=False, header = False)


In [35]:
# train_supervised require format just like the one in category_description
import fasttext

# train_supervise -> word embedding + classification; train_unsupervised = word embedding only
model = fasttext.train_supervised(input="dataset/ecommerce.train")
model.test("dataset/ecommerce.test")

Read 4M words
Number of words:  78910
Number of labels: 4
Progress: 100.0% words/sec/thread: 5160563 lr:  0.000000 avg.loss:  0.178668 ETA:   0h 0m 0s


(10085, 0.9677739216658403, 0.9677739216658403)

test_sample_size, precision, recall

In [36]:
model.predict("wintech assemble desktop pc cpu 500 gb sata hdd 4 gb ram intel c2d processor 3")

(('__label__electronics',), array([0.9989416]))

In [37]:
model.predict("ockey men's cotton t shirt fabric details 80 cotton 20 polyester super combed cotton rich fabric")


(('__label__clothing_accessories',), array([1.00001001]))

In [38]:
model.predict("think and grow rich deluxe edition")


(('__label__books',), array([1.00000989]))

In [39]:
model.get_nearest_neighbors("painting")

[(0.9985827803611755, 'wooden\xa0stand\xa0holdernote'),
 (0.9985827803611755, 'woodproduct'),
 (0.9985827803611755, 'setswooden'),
 (0.9985716938972473, 'moglix'),
 (0.9985716938972473, 'wcb32'),
 (0.998481273651123, '46l'),
 (0.9983208775520325, 'descriptionwant'),
 (0.9982855916023254, "box's"),
 (0.9982776045799255, 'qd2'),
 (0.9981309175491333, '1830005048')]

In [40]:
model.get_nearest_neighbors("sony")

[(0.9988576173782349, 'external\xa0hard\xa0disk'),
 (0.9987677931785583, 'jensen'),
 (0.9987677931785583, 'jta'),
 (0.9986138939857483, 'desprition'),
 (0.9986138939857483, 'ditail'),
 (0.9986138939857483, 'pr0duct'),
 (0.9985280632972717, 'hoverboard'),
 (0.9984050393104553, "headset's"),
 (0.9983731508255005, '76e500bw'),
 (0.9982991218566895, 'escalated')]